# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [1]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())


Dataset loaded successfully! Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning
*(Your code and analysis here. Think about standardization and how to handle the `Type` column!)*


In [ ]:
# ==========================================
# 1.1  INITIAL EXPLORATION
# ==========================================

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nShape: {df_raw.shape}")
print(f"\nColumn types:\n{df_raw.dtypes}")

print("\n" + "="*60)
print("FIRST 5 ROWS")
print("="*60)
display(df_raw.head())

print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)
display(df_raw.describe())

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

print("\n" + "="*60)
print("CLASS DISTRIBUTION – Machine failure (label, hidden later)")
print("="*60)
print(df_raw["Machine failure"].value_counts())
print(f"\nFailure rate: {df_raw['Machine failure'].mean()*100:.2f}%")


In [ ]:
# ==========================================
# 1.2  FEATURE SEPARATION
# ==========================================
# UDI and Product ID are identifiers – useless for anomaly detection.
# Machine failure + failure-mode flags (TWF, HDF, PWF, OSF, RNF) are
# LABELS we must hide during unsupervised training and bring back in Part 4.

label_cols = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
id_cols    = ["UDI", "Product ID"]

y_true = df_raw["Machine failure"].copy()

df = df_raw.drop(columns=id_cols + label_cols)
print(f"Working feature set: {list(df.columns)}")
print(f"Shape after dropping IDs & labels: {df.shape}")
display(df.head())

In [ ]:
# ==========================================
# 1.3  DISTRIBUTION ANALYSIS – Skewness & Histograms
# ==========================================

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(1, len(numerical_cols), figsize=(20, 4))
for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=50, edgecolor="k", alpha=0.7, color="steelblue")
    axes[i].set_title(col, fontsize=10)
    skew_val = df[col].skew()
    axes[i].annotate(f"skew = {skew_val:.2f}", xy=(0.05, 0.90),
                     xycoords="axes fraction", fontsize=9, color="red")
fig.suptitle("Feature Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nSkewness summary:")
for col in numerical_cols:
    s = df[col].skew()
    tag = "✓ OK" if abs(s) < 1 else "⚠ Skewed"
    print(f"  {col:35s}  skew={s:+.2f}  {tag}")

In [ ]:
# ==========================================
# 1.4  CORRELATION MATRIX
# ==========================================

fig, ax = plt.subplots(figsize=(8, 6))
corr = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Pearson Correlation Matrix (Numerical Features)")
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("• Air temperature & Process temperature are positively correlated (~0.88).")
print("  → This is expected: the process heats up proportionally to ambient temp.")
print("• Torque and Rotational speed show negative correlation (~−0.88).")
print("  → Higher torque → lower RPM (physics of the motor).")
print("• Tool wear is largely independent of the other sensors.")

In [ ]:
# ==========================================
# 1.5  BOX-PLOTS – quick outlier visual check
# ==========================================

fig, axes = plt.subplots(1, len(numerical_cols), figsize=(20, 4))
for i, col in enumerate(numerical_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="steelblue")
    axes[i].set_title(col, fontsize=10)
fig.suptitle("Box-plots of Numerical Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 1.6  CATEGORICAL FEATURE – Type
# ==========================================

print("Unique values in 'Type':", df["Type"].unique())
print("\nValue counts:")
print(df["Type"].value_counts())

fig, ax = plt.subplots(figsize=(5, 3))
df["Type"].value_counts().plot.bar(ax=ax, color=["#4e79a7", "#f28e2b", "#e15759"],
                                   edgecolor="k")
ax.set_title("Machine Type Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 1.7 — Preprocessing: Why Standardisation Matters

**Distance-based methods (LOF, Elliptic Envelope, OneClassSVM with RBF kernel):**

These algorithms measure proximity between data points — either through Euclidean distances (LOF), Mahalanobis distances (Elliptic Envelope), or kernel-space dot products (SVM/RBF).  
If features live on vastly different scales (e.g. *Rotational speed* ≈ 1300–2900 rpm vs. *Torque* ≈ 3–77 Nm), the high-magnitude feature will dominate the distance computation and the smaller-magnitude features will be effectively ignored.  
**Standardisation (zero mean, unit variance) puts every feature on an equal footing**, ensuring each one contributes proportionally to the distance metric.

**Isolation Forest — why scaling is *not* strictly required:**

Isolation Forest isolates anomalies by randomly selecting a feature and then randomly selecting a split value between the min and max of that feature. Because it only performs **axis-aligned splits and compares values within a single feature at a time**, the absolute scale of each feature is irrelevant — a split on *Rotational speed* (large numbers) never interacts arithmetically with a split on *Torque* (small numbers).  
Standardisation therefore does not change the tree structure or the anomaly scores. It is harmless to apply, but not necessary.

---

### 1.8 — Handling the Categorical Feature `Type`

The `Type` column takes three values: **L** (Low), **M** (Medium), **H** (High quality).  

For **distance-based algorithms** we cannot feed in a raw categorical string — distances are undefined on strings. Two common strategies:

| Strategy | Pros | Cons |
|---|---|---|
| **One-Hot Encoding** (3 binary columns) | No artificial ordering; works with any algorithm | Adds dimensions; Hamming-like distances on binary axes |
| **Ordinal Encoding** (H→0, M→1, L→2) | Single column; preserves a natural quality ordering | Assumes equal spacing between levels |

We choose **One-Hot Encoding** because it avoids imposing an arbitrary numeric gap between quality levels, and the extra 2 dimensions are negligible relative to the 5 numerical features. The one-hot columns are left unscaled (they are already 0/1) so distance-based methods treat them as equal-weight binary indicators.

In [ ]:
# ==========================================
# 1.9  PREPROCESSING PIPELINE
# ==========================================
from sklearn.preprocessing import StandardScaler

# --- One-Hot Encode 'Type' ---
df_encoded = pd.get_dummies(df, columns=["Type"], prefix="Type", drop_first=False)
print("Columns after one-hot encoding:", list(df_encoded.columns))

# --- Standardise numerical features only ---
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

print("\nAfter standardisation (first 5 rows):")
display(df_scaled.head())
print(f"\nFinal preprocessed shape: {df_scaled.shape}")
print("Means (should be ~0):\n", df_scaled[numerical_cols].mean().round(6).to_string())
print("\nStds  (should be ~1):\n", df_scaled[numerical_cols].std().round(4).to_string())

In [ ]:
# ==========================================
# 1.10  QUICK SANITY CHECK – Pair-plot on scaled data
# ==========================================

sample = df_scaled[numerical_cols].sample(n=min(1000, len(df_scaled)), random_state=42)
g = sns.pairplot(sample, diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10})
g.figure.suptitle("Pair-plot of Scaled Numerical Features (1 000-sample)", y=1.02)
plt.show()

print("\n✅ Part 1 complete — preprocessed DataFrame 'df_scaled' is ready for modelling.")

---
## Part 2: Modeling & Hyperparameter Tuning
*(Train IsolationForest, OneClassSVM, LocalOutlierFactor, and EllipticEnvelope. Remember to tune your threshold parameters!)*


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

# Your code here


---
## Part 3: Technical Comparison & Visualizations
*(Use PCA/t-SNE to visualize the flagged anomalies. Find an anomaly caught by one model but missed by another and explain why.)*


In [ ]:
from sklearn.decomposition import PCA

# Your code here


---
## Part 4: Managerial Conclusion & Business Strategy
*(Bring back `y_true`. Calculate the number of False Positives and False Negatives for each model. Apply the cost matrix. Which model wins?)*


In [ ]:
# Business Cost Matrix
COST_FP = 500     # False Positive: Wasted technician check
COST_FN = 15000   # False Negative: Catastrophic machine breakdown

# Your code here
